# Notebook 02 — Fine-tune EEGNet on MIMED (Epoc X)

This notebook implements **Step 2** of the transfer learning pipeline:
- Load the pretrained EEGNet checkpoint from Notebook 01 (trained on BCI IV 2a, 22ch)
- Adapt it to MIMED / EMOTIV Epoc X hardware (14ch) via partial weight transfer
- Evaluate using Leave-One-Trial-Out (LOTO) cross-validation
- Compare against a from-scratch baseline to validate the transfer learning benefit

## Transfer Learning Strategy


|  | Pretrained (BCI IV 2a, 22ch) | Fine-tuned (MIMED, 14ch)
|--|------------------------------|-------------------------|
| Temporal Conv |  ✅ load & freeze  |  Preserves learned ERD/ERS frequency features |  
| Spatial Conv  |  ❌ reinitialise   |  Must relearn: 22ch → 14ch, different layout  |
| Classifier    |  ❌ reinitialise   |  2-class output (Left / Right Hand only)      |


**Environment:** Google Colab T4 GPU

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
PROJECT_PATH = '/content/drive/MyDrive/Honours Project'
os.makedirs(os.path.join(PROJECT_PATH, 'results'), exist_ok=True)
os.chdir(PROJECT_PATH)

In [4]:
!ls

bciv2a.npz			   eegnet_pretrained_500.pth  results
eegnet_pretrained_500_new.pth	   MIMED
eegnet_pretrained_500_nonstop.pth  mimed.npz


In [5]:
!pip install numpy torch braindecode scikit-learn -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 497.9/497.9 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 113.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.4/178.4 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.5/268.5 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 127.6 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatibl

In [6]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from braindecode.models import EEGNet
from sklearn.metrics import accuracy_score, cohen_kappa_score
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


## 2. Load MIMED Data

Data was preprocessed in Notebook 00 (bandpass 8-30Hz, baseline corrected, z-score normalised).

**2-class only (Left / Right Hand)** — more tractable given the limited trial count per subject.

In [12]:
d     = np.load('mimed.npz', allow_pickle=True)
X_all = d['X'].astype(np.float32)
y_all = d['y'].astype(np.int64)

mask = y_all < 2
X    = X_all[mask]
y    = y_all[mask]

print(f'Full MIMED : {X_all.shape}')
print(f'2-class    : {X.shape} | classes: {np.unique(y)} | counts: {np.bincount(y)}')

Full MIMED : (1080, 14, 512)
2-class    : (600, 14, 512) | classes: [0 1] | counts: [480 120]


In [13]:
# Undersample class 0 to match class 1 (Left Hand 480 vs Right Hand 120)
idx_0         = np.where(y == 0)[0]
idx_1         = np.where(y == 1)[0]
idx_0_sampled = np.random.choice(idx_0, size=len(idx_1), replace=False)
idx_balanced  = np.sort(np.concatenate([idx_0_sampled, idx_1]))
X = X[idx_balanced]
y = y[idx_balanced]

print(f'Full MIMED : {X_all.shape}')
print(f'2-class    : {X.shape} | classes: {np.unique(y)} | counts: {np.bincount(y)}')

Full MIMED : (1080, 14, 512)
2-class    : (240, 14, 512) | classes: [0 1] | counts: [120 120]


## 3. Model Factory

In [14]:
def make_model(n_chans, n_outputs, n_times):
    return EEGNet(n_chans=n_chans, n_outputs=n_outputs, n_times=n_times, drop_prob=0.5).to(device)

## 4. Partial Weight Transfer

Load the pretrained checkpoint and copy only layers whose shapes match.

The temporal conv weights transfer cleanly — same kernel shape regardless of channel count.
The spatial conv (`n_channels, 1` kernel) and classifier head are reinitialised.

In [15]:
PRETRAINED_PATH = os.path.join(PROJECT_PATH, 'eegnet_pretrained_500_new.pth')

def load_partial_weights(model, pretrained_path):
    """Copy weights from pretrained checkpoint where shapes match. Skip the rest."""
    pretrained  = torch.load(pretrained_path, map_location=device)
    model_state = model.state_dict()
    transferred, skipped = [], []
    for name, param in pretrained.items():
        if name in model_state and model_state[name].shape == param.shape:
            model_state[name].copy_(param)
            transferred.append(name)
        else:
            skipped.append(name)
    model.load_state_dict(model_state)
    print(f'Transferred : {len(transferred)} layers')
    print(f'Skipped     : {len(skipped)} layers (shape mismatch)')
    return model

model_ft = make_model(n_chans=14, n_outputs=2, n_times=512)
model_ft = load_partial_weights(model_ft, PRETRAINED_PATH)

Transferred : 18 layers
Skipped     : 3 layers (shape mismatch)


## 5. Differential Learning Rates

The temporal layer is kept near-frozen (very low LR) to preserve pretrained ERD features.  
The spatial layer and classifier learn from scratch at a normal LR.

```
Temporal Conv  →  LR = 1e-5   (preserve pretrained frequency features)
Spatial Conv   →  LR = 1e-3   (relearn for 14ch Epoc X electrode layout)
Classifier     →  LR = 1e-3
```

In [16]:
def make_optimizer(model, lr_temporal=1e-5, lr_other=1e-3):
    temporal_params, other_params = [], []
    for name, param in model.named_parameters():
        if 'conv_temporal' in name or 'bnorm_temporal' in name:
            temporal_params.append(param)
        else:
            other_params.append(param)
    return torch.optim.Adam([
        {'params': temporal_params, 'lr': lr_temporal},
        {'params': other_params,    'lr': lr_other},
    ])

## 6. Training Utilities

In [18]:
loss_fn = nn.CrossEntropyLoss()

def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            preds   += model(X_batch.to(device)).argmax(dim=1).cpu().tolist()
            targets += y_batch.tolist()
    return accuracy_score(targets, preds), cohen_kappa_score(targets, preds)

## 7. Sliding Window Augmentation

MIMED has very few trials per subject (~8). Sliding window augmentation expands the
training set by extracting overlapping windows from each trial.

- Window: 512 timepoints (full 4s epoch)  
- Step: 128 (0.25s) → 75% overlap → 1 trial becomes ~4 trials  
- Applied **only on the training set** to avoid data leakage

In [19]:
def augment(X, y, noise_std=0.05):
    """
    Augment training data by adding Gaussian noise.
    Doubles the dataset size while preserving full 4s temporal context.
    """
    X_noisy = X + np.random.normal(0, noise_std, X.shape).astype(np.float32)
    X_aug   = np.concatenate([X, X_noisy])
    y_aug   = np.concatenate([y, y])
    return X_aug, y_aug

## 8. Leave-One-Trial-Out (LOTO) Cross-Validation

5-fold CV is not suitable here — MIMED has too few trials (~8/subject), so each fold  
would only test 1–2 trials. LOTO holds out one trial at a time for a more stable estimate.

Two variants are compared:
1. **From scratch** — random initialisation, trained on MIMED only  
2. **Fine-tuned** — partial weights from BCI IV 2a pretrained model

In [20]:
EPOCHS_FT = 150
BATCH     = 32

def run_loto(X, y, use_pretrained=True, label=''):
    results = []
    n = len(X)
    for i in range(n):
        train_idx = [j for j in range(n) if j != i]
        X_train_aug, y_train_aug = augment(X[train_idx], y[train_idx])

        train_loader = DataLoader(
            TensorDataset(torch.tensor(X_train_aug), torch.tensor(y_train_aug)),
            batch_size=BATCH, shuffle=True
        )
        test_loader = DataLoader(
            TensorDataset(torch.tensor(X[[i]]), torch.tensor(y[[i]])),
            batch_size=BATCH
        )

        model = make_model(n_chans=14, n_outputs=2, n_times=512)
        if use_pretrained:
            model     = load_partial_weights(model, PRETRAINED_PATH)
            optimizer = make_optimizer(model)
        else:
            optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=15, factor=0.5)

        for epoch in range(EPOCHS_FT):
            loss = train_epoch(model, train_loader, optimizer)
            scheduler.step(loss)

        acc, kappa = evaluate(model, test_loader)
        results.append((acc, kappa))
        if (i+1) % 20 == 0:
            print(f'  [{label}] Trial {i+1}/{n}')

    return results

In [21]:
print(X.shape)
print(y.shape)
print(np.bincount(y))

(240, 14, 512)
(240,)
[120 120]


In [22]:
print('Running from-scratch baseline...')
results_scratch = run_loto(X, y, use_pretrained=False, label='scratch')
accs_s   = [r[0] for r in results_scratch]
kappas_s = [r[1] for r in results_scratch]
print(f'From scratch  →  Acc: {np.mean(accs_s):.4f} ± {np.std(accs_s):.4f} | Kappa: {np.mean(kappas_s):.4f}')

Running from-scratch baseline...


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.

  [scratch] Trial 20/240


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.

  [scratch] Trial 40/240


...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.

  [scratch] Trial 220/240


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.

  [scratch] Trial 240/240
From scratch  →  Acc: 0.7167 ± 0.4506 | Kappa: nan


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)


In [23]:
print('Running fine-tuned model...')
results_ft = run_loto(X, y, use_pretrained=True, label='finetune')
accs_ft   = [r[0] for r in results_ft]
kappas_ft = [r[1] for r in results_ft]
print(f'Fine-tuned    →  Acc: {np.mean(accs_ft):.4f} ± {np.std(accs_ft):.4f} | Kappa: {np.mean(kappas_ft):.4f}')

Running fine-tuned model...
Transferred : 18 layers
Skipped     : 3 layers (shape mismatch)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)


Transferred : 18 layers
Skipped     : 3 layers (shape mismatch)
Transferred : 18 layers
Skipped     : 3 layers (shape mismatch)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)


Transferred : 18 layers
Skipped     : 3 layers (shape mismatch)


...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)


Transferred : 18 layers
Skipped     : 3 layers (shape mismatch)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)


Transferred : 18 layers
Skipped     : 3 layers (shape mismatch)
  [finetune] Trial 240/240
Fine-tuned    →  Acc: 0.7208 ± 0.4486 | Kappa: nan


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)


## 9. Results

In [24]:
df = pd.DataFrame({
    'Model':    ['EEGNet from scratch', 'EEGNet fine-tuned (BCI IV 2a pretrained)'],
    'Acc':      [f'{np.mean(accs_s):.4f} ± {np.std(accs_s):.4f}',
                 f'{np.mean(accs_ft):.4f} ± {np.std(accs_ft):.4f}'],
    'Kappa':    [f'{np.mean(kappas_s):.4f}', f'{np.mean(kappas_ft):.4f}'],
    'Protocol': ['LOTO, MIMED 2-class'] * 2,
})
print(df.to_string(index=False))
df.to_csv('results/finetune_mimed_results.csv', index=False)
print('\nSaved: results/finetune_mimed_results.csv')

                                   Model             Acc Kappa            Protocol
                     EEGNet from scratch 0.7167 ± 0.4506   nan LOTO, MIMED 2-class
EEGNet fine-tuned (BCI IV 2a pretrained) 0.7208 ± 0.4486   nan LOTO, MIMED 2-class

Saved: results/finetune_mimed_results.csv


In [26]:
from sklearn.metrics import cohen_kappa_score
import warnings

def safe_kappa(results):
    # results là list of (acc, kappa) — recalculate from stored preds
    kappas = []
    for acc, _ in results:
        # acc từ LOTO là 0.0 hoặc 1.0 (1 trial mỗi fold)
        # kappa trên 1 trial không có nghĩa thống kê → dùng mean acc thay thế
        kappas.append(acc)
    return np.mean(kappas)

# Kappa xấp xỉ từ accuracy (chỉ dùng khi test set = 1 trial)
kappa_s  = 2 * np.mean(accs_s)  - 1   # formula: kappa ≈ 2*acc - 1 cho binary
kappa_ft = 2 * np.mean(accs_ft) - 1

print(f'From scratch  →  Acc: {np.mean(accs_s):.4f} | Kappa (approx): {kappa_s:.4f}')
print(f'Fine-tuned    →  Acc: {np.mean(accs_ft):.4f} | Kappa (approx): {kappa_ft:.4f}')

From scratch  →  Acc: 0.7167 | Kappa (approx): 0.4333
Fine-tuned    →  Acc: 0.7208 | Kappa (approx): 0.4417


## 10. Save Fine-tuned Checkpoint

Retrain on 100% of MIMED data and save for use in Notebook 03.

In [25]:
X_aug_full, y_aug_full = augment(X, y)
full_loader = DataLoader(
    TensorDataset(torch.tensor(X_aug_full), torch.tensor(y_aug_full)),
    batch_size=BATCH, shuffle=True
)

model_final = make_model(n_chans=14, n_outputs=2, n_times=512)
model_final = load_partial_weights(model_final, PRETRAINED_PATH)
optimizer   = make_optimizer(model_final)
scheduler   = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=15, factor=0.5)

for epoch in range(EPOCHS_FT):
    loss = train_epoch(model_final, full_loader, optimizer)
    scheduler.step(loss)
    if (epoch+1) % 30 == 0:
        print(f'Epoch {epoch+1} | Loss: {loss:.4f}')

torch.save(model_final.state_dict(), os.path.join(PROJECT_PATH, 'eegnet_finetuned_mimed.pth'))
print('Saved: eegnet_finetuned_mimed.pth')

Transferred : 18 layers
Skipped     : 3 layers (shape mismatch)
Epoch 30 | Loss: 0.4102
Epoch 60 | Loss: 0.3458
Epoch 90 | Loss: 0.3231
Epoch 120 | Loss: 0.2628
Epoch 150 | Loss: 0.2542
Saved: eegnet_finetuned_mimed.pth
